## Lab 21 | AWS External Functions in Snowflake
**Snowflake Advanced 3-day class Lab · © 2026 Innovation In Software Corporation. All rights reserved.**

Topics covered:
1. Create and test an AWS Lambda function
2. Connect an IAM-protected API Gateway REST API
3. Create a Snowflake API integration and establish AWS trust
4. Call remote Python logic from SQL, including a batch and NULL input
5. Verify against the course orders; optionally add an FX conversion; clean up both platforms

**Run one cell at a time. Do not use Run All.** AWS console actions between cells are required. Execute Python cells in a Snowflake-hosted notebook with `snowflake-snowpark-python` available. The notebook generates policies and SQL; it does not use AWS credentials or deploy AWS resources automatically.

The request path is **SQL → external function / API integration → API Gateway → Lambda → SQL result**. This is an external function, not a Python UDF using an external access integration. The risk rule is deliberately simple; the lesson is the cross-platform connection, not predictive risk analysis.

**Prerequisites:** AWS permissions for Lambda, API Gateway REST APIs, IAM roles/policies and CloudWatch logs; Snowflake ACCOUNTADMIN for integration setup; existing course role `ADV_LAB`, warehouse `ADV_WH`, and database `ADV_ORDERS`. Use dedicated training resources. AWS services and Snowflake compute can incur charges.


**Workflow:** notebook preparation → AWS resources and empty invocation role → notebook integration and both policy documents → AWS policies → notebook function and checks → optional FX demo → final AWS log review and cleanup. Keep notebook and AWS tabs side by side for copying prepared text. No separate SQL worksheet, local Python environment or AWS CLI is required.


### Reset a previous run (skip on first use)
Use this section only if you already created this demo. It removes the previous demo so the same final names can be used from start to finish. Do not delete course database objects beyond the specific functions, temporary table and integration listed below.

**Snowflake:** execute the following SQL cell to remove the previous demo functions and integration. Skip this cell on a first run.


In [ ]:
%%sql
USE ROLE ACCOUNTADMIN;
DROP FUNCTION IF EXISTS ADV_ORDERS.OPS.GET_FX_RATE(VARCHAR, VARCHAR);
DROP FUNCTION IF EXISTS ADV_ORDERS.OPS.DEMO_ORDER_RISK(FLOAT);
DROP INTEGRATION IF EXISTS ORDER_RISK_API_INTEGRATION;

**Optional temporary-table reset:** A temporary table belongs only to the session that created it and disappears when that session ends. If you ran the FX extension in this same session and have not already removed its table, run the next cell; otherwise skip it. It switches back to the table's owner, ADV_LAB, and targets only the FX table created by the optional extension. Never run it against an unrelated permanent table of the same name.


In [ ]:
USE ROLE ADV_LAB;
DROP TABLE IF EXISTS ADV_ORDERS.OPS.LAB21_FX_USD_EUR;


**AWS console**, in the region used for the previous run (use the region selected for your demo):
1. API Gateway: find `snowflake-order-risk-api`, verify its identity, and delete the REST API and its stages. Confirm the API ID belongs to this demo before deleting it.
2. Lambda: open `snowflake-order-risk-demo`. Record its execution role under **Configuration → Permissions**, then use **Actions → Delete** to remove the function.
3. IAM: delete `SnowflakeOrderRiskInvokeRole` and its inline invocation policy. Delete the recorded Lambda execution role only if it was created solely for this demo and is not shared. Do not delete AWS-managed policies; remove a dedicated customer-managed logging policy only if it is unused.
4. CloudWatch Logs: delete `/aws/lambda/snowflake-order-risk-demo` if the previous logs are no longer needed.

**CHECKPOINT:** the demo API, Lambda, dedicated IAM roles, Snowflake function and API integration have been removed. Now start Step 1 using the final names. The newly created API URL and Snowflake-generated external ID must be obtained again during this run.

For cleanup after a completed run, use the final cleanup section, which uses that run's configured names.


### STEP 1 | Prepare names, code and test payloads in the notebook
Run the setup cells, then use the static Lambda code and test JSON blocks below. Keep this section beside the AWS console; no notebook execution is needed during the first AWS visit.

Use the same final names on every run. If resources already exist, complete the reset section first. Enter your AWS account ID as a string to preserve leading zeros. Python generates the JSON and parameterized SQL; SQL executes directly through this notebook's Snowflake session.


In [ ]:
%%sql
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE ADV_WH;
USE DATABASE ADV_ORDERS;
USE SCHEMA OPS;
ALTER SESSION SET USE_CACHED_RESULT = FALSE;
SELECT CURRENT_ROLE(), CURRENT_REGION(), CURRENT_WAREHOUSE();

In [ ]:
import ipywidgets as widgets
from IPython.display import display

account_input = widgets.Text(
    description="AWS Account ID:",
    placeholder="12-digit account ID",
    style={"description_width": "initial"}
)

display(account_input)

In [ ]:
import json
import re
from urllib.parse import urlparse
from snowflake.snowpark.context import get_active_session

session = get_active_session()

aws_account_id = account_input.value.strip()

assert len(aws_account_id) == 12 and aws_account_id.isdigit()

aws_region = "us-east-2"
lambda_name = "snowflake-order-risk-demo"
api_name = "snowflake-order-risk-api"
aws_role_name = "SnowflakeOrderRiskInvokeRole"
inline_policy_name = "InvokeOrderRiskDemo"
stage_name = "demo"
integration_name = "ORDER_RISK_API_INTEGRATION"
function_name = "ADV_ORDERS.OPS.DEMO_ORDER_RISK"

assert re.fullmatch(r"\d{12}", aws_account_id)
assert re.fullmatch(r"[A-Z][A-Z0-9_]*", integration_name)
assert re.fullmatch(r"[A-Z][A-Z0-9_]*(\.[A-Z][A-Z0-9_]*){2}", function_name)

def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"

print("Lambda:", lambda_name)
print("API Name:", api_name)
print("IAM role:", aws_role_name)
print("Snowflake function:", function_name)


**Lambda code — copy into `lambda_function.py`:**

```python
import json

def lambda_handler(event, context):
    rows = json.loads(event["body"])["data"]
    results = []

    for row_number, amount in rows:
        if amount is None:
            risk = "UNKNOWN"
        elif amount > 100:
            risk = "REVIEW"
        else:
            risk = "OK"
        results.append([row_number, risk])

    return {
        "statusCode": 200,
        "headers": {"Content-Type": "application/json"},
        "body": json.dumps({"data": results})
    }
```

**Lambda test event:**

```json
{
  "body": "{\"data\": [[0, 120], [1, 45], [2, null]]}"
}
```

**API Gateway request body:**

```json
{
  "data": [
    [0, 120], [1, 45], [2, null]
  ]
}
```

### STEP 2 | Create the Lambda function in AWS
In the AWS console, select the configured region and open **Lambda**.

1. On the welcome page, click **Create an Event Function**. The “How it works” editor is only an introductory example.
2. Choose **Author from scratch**.
3. Enter the Lambda name printed above; select **Python 3.12** in the **Runtime** drop-down box.
4. Leave **Durable execution** and **EC2 capacity provider** off. Leave **Additional settings** at their defaults.
5. Click **Create Event Function**; dismiss the **Getting started** popup.
6. Open **Code → lambda_function.py**. Replace the starter code with the Lambda code prepared in Step 1, then click **Deploy** in the editor's left panel.

This execution role lets Lambda write logs. A separate IAM role will later authorize Snowflake to invoke API Gateway.


### STEP 3 | Test Lambda before connecting the gateway
1. Under **TEST EVENTS [NONE SELECTED]**, click **Create new test event**.
2. Name it `snowflake_batch_test`; choose **Private**.
3. Replace Event JSON with the **LAMBDA TEST EVENT** prepared in Step 1.
4. Save the event. If the save action is not visible, scroll **inside the right-hand test-event form**, outside the JSON editor; the form has its own scrollbar.
5. Click the blue **Test** button below Deploy.

**CHECKPOINT:** statusCode `200`; response body contains `[[0,"REVIEW"],[1,"OK"],[2,"UNKNOWN"]]` under `data`. The body is a JSON-encoded string, so escaped quotes are normal.

Snowflake sends batches even though SQL expresses one function result per row. The first item in each input/output row is a **batch row number**, not an order ID. Return one result per input, preserving those numbers and order. Three inputs simply exercise the three branches; batch size is not fixed at three. Keep the handler free of side effects because remote requests can be retried.


### STEP 3.5 | Record the Lambda execution role

AWS automatically created an execution role for this function.
Our cleanup script will use its name to remove the role at the end.

1. Open the Lambda function's **Configuration** tab.
2. Select **Permissions**.
3. Under **Execution role**, copy the **role name**, not its ARN.
4. Paste it into the Python cell below.

In [ ]:
lambda_execution_role = "snowflake-order-risk-demo-role-pt56mg27"  # "PASTE_LAMBDA_EXECUTION_ROLE_NAME_HERE"

### STEP 4 | Create the API Gateway REST API
1. Search for **API Gateway** in the AWS console. Keep the same region as Lambda.
2. Choose **Create an API → REST API → Build** (not HTTP API or REST API Private).
3. Select **New API**, enter the configured API name, and leave Description optional.
4. Set **Endpoint type = Regional** and **IP address type = IPv4**.
5. Select **SecurityPolicy_TLS13_1_2_2021_06**, supporting TLS 1.2 and TLS 1.3.
6. Click **Create API**.

The TLS policy controls transport encryption; IAM authorization is configured separately below.


### STEP 5 | Add the root POST method
On the Resources page, keep the root `/` selected and click **Create method**.

| Setting | Value |
|---|---|
| Method type | POST |
| Integration type | Lambda function |
| Lambda proxy integration | Enabled |
| Response transfer mode | Buffered |
| Lambda region | The configured region |
| Lambda function | The function created in Step 2 |
| Integration timeout | 29000 ms (default) |

Expand **Method request settings**:
- Authorization: **AWS IAM**
- Request validator: **None**
- API key required: **unchecked**
- Operation name: leave blank (the displayed `GetPets` text is a placeholder)

Leave URL query strings, HTTP headers and request-body configuration unchanged. Click **Create method**. The console grants API Gateway permission to invoke Lambda through Lambda's resource-based policy.

**CHECKPOINT:** the POST Method execution page shows **Authorization = AWS_IAM**. Lambda proxy integration passes the HTTP body to our handler as `event["body"]`.


### STEP 6 | Test API Gateway, then deploy
1. On the POST method page, select **Test**, to the right of Method response.
2. Paste the **API Gateway request body** prepared in Step 1 into **Request body**; leave other optional fields empty.
3. Click **Test**.

Unlike the Lambda test, this is the raw HTTP body: do not add the outer `body` wrapper. API Gateway supplies that wrapper.

**CHECKPOINT:** HTTP `200` and the same REVIEW / OK / UNKNOWN results. This proves gateway-to-Lambda connectivity, not the Snowflake IAM trust relationship.

4. Click **Deploy API** at the upper right.
5. Choose **New stage**, name it `demo`, and click **Deploy**.
6. Copy the stage's **Invoke URL** for Step 8. Stay in AWS and create the IAM role in Step 7 before returning to the notebook. We use the root resource, so no additional resource path is appended.

An API Gateway stage is a deployment environment; it is unrelated to a Snowflake file stage. Changes to methods or API policies may require redeployment to this stage.


In [ ]:
invoke_url = "https://wa8ukluzkb.execute-api.us-east-2.amazonaws.com/demo" ""

### STEP 7 | Create the invocation role before leaving AWS
1. Open **IAM → Roles → Create role**.
2. Choose **AWS account → This account** as the temporary trusted entity.
3. Click **Next**; select no permission policies and click **Next** again.
4. Enter `SnowflakeOrderRiskInvokeRole` in the **Role name** box (or the configured role name) and click **Create role**.
5. Record its ARN from the summary page.

Do not add the invocation policy yet. We will generate both final policies together after Snowflake creates its integration identity. The new role has no API invocation permissions at this point.

**Return to the notebook now**, with the deployed Invoke URL and the role ARN. This ends the first AWS visit.


In [ ]:
aws_role_arn_from_console = "arn:aws:iam::024855656562:role/SnowflakeOrderRiskInvokeRole".strip()  # PASTE_ROLE_ARN_HERE

### STEP 8 | Record the endpoint and verify the role ARN
Paste the Invoke URL from this run below. A recreated API receives a new URL. The cell constructs the exact POST permission and role ARN; compare the printed role ARN with the AWS role summary. If they differ, correct the configured account ID or role name before continuing.


In [ ]:

parsed = urlparse(invoke_url)
assert parsed.scheme == "https", "Paste the deployed API Invoke URL."
assert parsed.hostname and parsed.hostname.endswith(f".execute-api.{aws_region}.amazonaws.com")
assert parsed.path.rstrip("/") == f"/{stage_name}"
assert not parsed.query and not parsed.fragment
invoke_url = invoke_url.rstrip("/")
api_id = parsed.hostname.split(".")[0]
aws_role_arn = f"arn:aws:iam::{aws_account_id}:role/{aws_role_name}"
invoke_arn = f"arn:aws:execute-api:{aws_region}:{aws_account_id}:{api_id}/{stage_name}/POST/"
invoke_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Action": "execute-api:Invoke", "Resource": invoke_arn}]
}
print("IAM role:", aws_role_name)
print("Policy name:", inline_policy_name)
print(json.dumps(invoke_policy, indent=2))

assert aws_role_arn_from_console == aws_role_arn, "IAM role ARN mismatch. Check the AWS account ID and role name."


### STEP 9 | Create the Snowflake API integration
The integration references the IAM role and limits the permitted URL prefix. Run this cell once per fresh run. It intentionally uses **CREATE**, not CREATE OR REPLACE: replacing an integration can change the external ID and require updating AWS trust again.

If the object already exists, use the following DESCRIBE cell to inspect it, or finish cleanup before starting a fresh run.


In [ ]:
%%sql
USE ROLE ACCOUNTADMIN;

In [ ]:
integration_ddl = f"""
CREATE API INTEGRATION {integration_name}
    API_PROVIDER = aws_api_gateway
    API_AWS_ROLE_ARN = {sql_literal(aws_role_arn)}
    API_ALLOWED_PREFIXES = ({sql_literal(invoke_url)})
    ENABLED = TRUE
"""
    
print(integration_ddl)
session.sql(integration_ddl).collect()
print("API integration created.")


In [ ]:
properties = {
    row["property"]: row["property_value"]
    for row in session.sql(f"DESCRIBE INTEGRATION {integration_name}").collect()
}

sf_iam_user_arn = properties["API_AWS_IAM_USER_ARN"]
external_id = properties["API_AWS_EXTERNAL_ID"]

print("API_AWS_IAM_USER_ARN:", sf_iam_user_arn)
print("API_AWS_EXTERNAL_ID:", external_id)

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"AWS": sf_iam_user_arn},
        "Action": "sts:AssumeRole",
        "Condition": {"StringEquals": {"sts:ExternalId": external_id}}
    }]
}


In [ ]:

print("\nINLINE INVOCATION POLICY — name:", inline_policy_name)
print(json.dumps(invoke_policy, indent=2))

In [ ]:
print("\nReplace the IAM role trust policy with:")
print(json.dumps(trust_policy, indent=2))

### STEP 10 | Apply both policies in one AWS visit
Open **IAM → Roles → your invocation role**.

1. Under **Permissions → Add permissions → Create inline policy → JSON**, paste the **INLINE INVOCATION POLICY** printed above.
2. Click **Next**, name it `InvokeOrderRiskDemo` (or the configured policy name), and click **Create policy** button.
3. Open **Trust relationships → Edit trust policy**.
4. Replace the entire temporary trust policy with the **trust policy** printed above, and click **Update policy**.

The permissions policy grants invocation of only this API/stage/POST/root path. The trust policy allows the exact Snowflake principal to assume the role with the generated external ID. Do not modify Lambda's execution role.

**CHECKPOINT:** both policies are saved. Return to the notebook for Steps 11–14. If you choose the optional FX extension after Step 14, return to AWS for its checkpoints; otherwise the next visit is for final log inspection and cleanup. IAM changes may take a short time to propagate.


### STEP 11 | Create the external function
Use the existing course role for function ownership and execution. The definition is stored in Snowflake; its Python implementation runs in AWS Lambda. `CALLED ON NULL INPUT` makes the NULL branch explicit. `VOLATILE` reflects that the remote implementation can change.


In [ ]:
%%sql
USE ROLE ACCOUNTADMIN;
GRANT USAGE ON INTEGRATION ORDER_RISK_API_INTEGRATION TO ROLE ADV_LAB;
USE ROLE ADV_LAB;
USE SECONDARY ROLES NONE;
USE WAREHOUSE ADV_WH;

In [ ]:
function_ddl = f"""
CREATE OR REPLACE EXTERNAL FUNCTION {function_name}(AMOUNT FLOAT)
RETURNS VARCHAR
CALLED ON NULL INPUT
VOLATILE
API_INTEGRATION = {integration_name}
AS {sql_literal(invoke_url)}
"""

print(function_ddl)
session.sql(function_ddl).collect()
print("External function created.")


### STEP 12 | Verify the complete connection
The next query tests above-threshold, below-threshold, boundary, and NULL values. The client row IDs in VALUES make the output easy to check; they are not the internal batch row numbers sent over HTTP.

**CHECKPOINT:** 120 → REVIEW, 45 → OK, 100 → OK, NULL → UNKNOWN. A successful Lambda-console or API-console test alone is not sufficient; this SQL test verifies Snowflake authentication and the full request path.


In [ ]:
SELECT TEST_ID, 
       AMOUNT, 
       ADV_ORDERS.OPS.DEMO_ORDER_RISK(amount::FLOAT) AS RISK_LABEL,
       EXPECTED_LABEL,
       IFF(RISK_LABEL = EXPECTED_LABEL, 'PASS', 'FAIL') AS verfication
FROM 
VALUES (1, 120, 'REVIEW'),
       (2, 45, 'OK'),
       (3, 100, 'OK'), 
       (4, NULL, 'UNKNOWN') AS orders (test_id, amount, expected_label)
ORDER BY TEST_ID;

### STEP 13 | Apply the function to course orders
This is a read-only demonstration. No pipeline table or task is modified. The query compares the remote output with our deliberately simple local SQL rule, so it does not depend on fixed order amounts.


In [ ]:
SELECT ORDER_ID, 
       AMOUNT,
       ADV_ORDERS.OPS.DEMO_ORDER_RISK(AMOUNT::FLOAT) AS RISK_LABEL,
       CASE WHEN AMOUNT IS NULL THEN 'UNKNOWN'
            WHEN AMOUNT > 100 THEN 'REVIEW' 
            ELSE 'OK' 
       END AS EXPECTED_LABEL,
       IFF(RISK_LABEL = EXPECTED_LABEL, 'PASS', 'FAIL') AS verification
FROM ADV_ORDERS.SILVER.ORDERS_CURRENT
ORDER BY ORDER_ID;

### STEP 14 | Explain the execution boundary
The SQL checkpoints above are the end-to-end checkpoint. Stay in the notebook here; inspect CloudWatch logs during the final AWS cleanup visit below. The default handler does not print input data, and Snowflake can batch rows, so invocation counts need not equal row counts.

Discussion:
- Snowflake manages SQL execution and authenticates through the API integration.
- API Gateway checks IAM authorization and forwards the request.
- Lambda runs the Python rule and returns batched results.
- Network calls introduce latency and failure modes. Use this for remote capabilities/services, not as a preferred replacement for a simple SQL CASE.

**Troubleshooting:**
- Access denied while creating the integration: check the Snowflake role.
- Cannot assume role: compare the IAM trust principal and external ID with DESCRIBE INTEGRATION; allow IAM propagation time.
- HTTP 403: check API authorization, invocation role permissions, API/stage/method ARN, and deployed URL.
- Missing Authentication Token: check stage/path and method; a browser issues GET but this API exposes POST.
- HTTP 500/502: check Lambda logs, proxy integration, and the response's string-valued `body`.
- KeyError `body` in Lambda: distinguish the wrapped Lambda test event from the raw API Gateway request body.
- After changing API methods/policies, redeploy to `demo` before testing again.


### OPTIONAL STEP 14A | Extend the same Lambda for currency conversion
Complete Steps 1–14 first. Skip to Step 15 if you do not want the extension. Finish the AWS checkpoints and the Snowflake queries below before cleanup.

Assume the course order `AMOUNT` is USD **for this demonstration**; the source does not establish a currency here. Frankfurter returns reference rates, not live trading quotes. The result date may precede today. SQL multiplies the amount; API Gateway authenticates and routes; Lambda adapts Snowflake batches to Frankfurter's HTTPS REST API. Snowflake external network access is not used.

Reuse the existing Lambda, REST API, `demo` stage, invocation role and `ORDER_RISK_API_INTEGRATION`. Do not create another function, integration, or trust policy. The original root POST and risk threshold remain unchanged.

**AWS checkpoint — replace and deploy the Lambda code:** Open the existing function → **Code → lambda_function.py**. Replace all code with the following and click **Deploy**. Then **Configuration → General configuration → Edit**: set **Timeout = 15 seconds** and save. The outbound socket timeout is 6 seconds, leaving room for processing within the Lambda timeout, which is below the existing 29-second API Gateway integration timeout. A socket timeout is not a total request deadline; Lambda is the overall limit. This lab's FX function limits batches to one row; a large batch with many distinct pairs could exceed the Lambda timeout.

The handler uses only the Python standard library. It accepts a Snowflake-format batch, caches duplicate pairs for one invocation, preserves row numbers/order, and returns each complete Frankfurter object. Invalid codes and upstream/network/response failures return an explicit HTTP error and fail the SQL statement rather than inventing a rate. Frankfurter validates whether a syntactically valid code is supported.


**Replacement `lambda_function.py` — supports both risk and FX:**

```python
import json
import math
import re
from datetime import date
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


def response(status, payload):
    return {
        "statusCode": status,
        "headers": {"Content-Type": "application/json"},
        "body": json.dumps(payload),
    }


def fx_rate(base, quote):
    if not isinstance(base, str) or not re.fullmatch(r"[A-Za-z]{3}", base):
        raise ValueError("Base currency must be a three-letter ASCII code.")
    if not isinstance(quote, str) or not re.fullmatch(r"[A-Za-z]{3}", quote):
        raise ValueError("Quote currency must be a three-letter ASCII code.")
    base, quote = base.upper(), quote.upper()
    url = f"https://api.frankfurter.dev/v2/rate/{base.lower()}/{quote.lower()}"
    request = Request(url, headers={"Accept": "application/json"})
    with urlopen(request, timeout=6) as upstream:
        result = json.load(upstream)
    if not isinstance(result, dict):
        raise ValueError("Frankfurter response is not an object.")
    rate = result.get("rate")
    if (result.get("base") != base or result.get("quote") != quote
            or isinstance(rate, bool) or not isinstance(rate, (int, float))
            or not math.isfinite(rate) or rate <= 0):
        raise ValueError("Frankfurter response has an invalid pair or rate.")
    try:
        date.fromisoformat(result["date"])
    except (KeyError, TypeError, ValueError) as exc:
        raise ValueError("Frankfurter response has an invalid rate date.") from exc
    return result


def lambda_handler(event, context):
    resource = event.get("resource")
    path = event.get("path")
    stage = (event.get("requestContext") or {}).get("stage")
    if resource is not None:
        route = resource
    elif stage and path and path.startswith("/" + stage + "/"):
        route = path[len(stage) + 1:]
    else:
        route = path or "/"
    if route not in ("/", "/fx"):
        return response(404, {"error": "Unknown API resource."})
    try:
        rows = json.loads(event["body"])["data"]
        if not isinstance(rows, list):
            raise ValueError("data must be an array.")
        results = []
        cache = {}
        for row in rows:
            if not isinstance(row, list) or len(row) != (3 if route == "/fx" else 2):
                raise ValueError("Invalid batch row shape.")
            if route == "/fx":
                row_number, base, quote = row
                if (not isinstance(base, str) or not re.fullmatch(r"[A-Za-z]{3}", base)
                        or not isinstance(quote, str) or not re.fullmatch(r"[A-Za-z]{3}", quote)):
                    raise ValueError("Currency codes must be three-letter ASCII codes.")
                pair = (base.upper(), quote.upper())
                if pair not in cache:
                    cache[pair] = fx_rate(base, quote)
                result = cache[pair]
            else:
                row_number, amount = row
                if amount is None:
                    result = "UNKNOWN"
                elif amount > 100:
                    result = "REVIEW"
                else:
                    result = "OK"
            results.append([row_number, result])
        return response(200, {"data": results})
    except HTTPError as exc:
        print(f"Frankfurter HTTP error: {exc.code}")
        return response(502, {"error": f"Frankfurter returned HTTP {exc.code}."})
    except (URLError, TimeoutError, OSError) as exc:
        print(f"Frankfurter network error: {exc}")
        return response(502, {"error": "Frankfurter request failed or timed out; inspect Lambda logs."})
    except (KeyError, TypeError, ValueError) as exc:
        print(f"Invalid request or upstream response: {exc}")
        return response(400, {"error": str(exc)})
```


**AWS checkpoint — test the replacement Lambda:** Run the original `snowflake_batch_test` from Step 3 again. With no routing metadata it must still return REVIEW / OK / UNKNOWN.

Create and run a second Lambda test event:
```json
{
  "resource": "/fx",
  "path": "/fx",
  "body": "{\"data\": [[0, \"USD\", \"EUR\"], [1, \"usd\", \"eur\"]]}"
}
```

**CHECKPOINT:** `statusCode` is `200`; the string-valued body decodes to the shape below. The rate and date are **illustrative**, not expected values; both rows should contain the same object because the duplicate pair is fetched once in this invocation.
```json
{"data": [[0, {"date": "2026-01-02", "base": "USD", "quote": "EUR", "rate": 0.9}], [1, {"date": "2026-01-02", "base": "USD", "quote": "EUR", "rate": 0.9}]]}
```
For an unsupported code or network failure, inspect the error response and Lambda CloudWatch logs; do not treat it as a valid rate. A Lambda test does not verify API Gateway or Snowflake authorization.


### OPTIONAL STEP 14B | Prepare the existing role's expanded policy
Run the next cell. It prints the FX endpoint and a **replacement** for the existing inline policy, allowing only the existing root POST and the new FX POST. Both ARNs and the URL are derived from this run's validated variables. Keep the existing Snowflake trust policy and API integration unchanged.


In [ ]:
fx_url = invoke_url + "/fx"
fx_invoke_arn = invoke_arn + "fx"
fx_invoke_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": "execute-api:Invoke",
        "Resource": [invoke_arn, fx_invoke_arn]
    }]
}
print("FX endpoint:", fx_url)
print("Replace inline policy:", inline_policy_name)
print(json.dumps(fx_invoke_policy, indent=2))


### OPTIONAL STEP 14C | Add `/fx` to the existing REST API
**AWS checkpoint — create the method:** In API Gateway select the same REST API and its root `/`. Choose **Create resource**, resource name/path `fx`, and create it. Select `/fx` → **Create method**. Configure **POST**, Lambda function integration, **Lambda proxy integration enabled**, **Buffered**, the same region and existing Lambda function, integration timeout **29000 ms**, and method authorization **AWS IAM**; leave API key required unchecked and request validator None. Create the method and accept the Lambda invocation permission prompt if shown. Do not remove or change root POST.

**CHECKPOINT:** `/fx` POST shows `AWS_IAM` and the same Lambda proxy integration; root `/` POST still exists with `AWS_IAM`. In `/fx` POST → **Test**, paste this raw request body (not the Lambda event wrapper):
```json
{"data": [[0, "USD", "EUR"], [1, "USD", "EUR"]]}
```
Expect HTTP `200` and the illustrative response shape from Step 14A, with the current rate/date. The console test checks integration, not the Snowflake role's signed request.

**AWS checkpoint — authorize and deploy:** In IAM → Roles → the existing invocation role → Permissions → the existing inline policy → **Edit → JSON**, replace its JSON with the policy printed above and save. Confirm it includes both exact `demo/POST/` and `demo/POST/fx` ARNs, with no wildcard. Do not edit the trust relationship or Lambda execution role. Back in API Gateway choose **Deploy API**, select the existing `demo` stage, and deploy. IAM changes may take time to propagate.

The integration's existing `API_ALLOWED_PREFIXES` contains the stage URL; the printed `/fx` URL is beneath that prefix. No integration replacement or trust-policy change is needed. A deployed Snowflake test below verifies the complete authenticated path.


### OPTIONAL STEP 14D | Create and test the Snowflake FX function
Use the existing `ADV_LAB` role and integration usage grant from Step 11. `RETURNS NULL ON NULL INPUT` avoids a remote call for a NULL argument; syntactically invalid non-NULL codes fail explicitly. `VOLATILE` reflects changing reference rates. `MAX_BATCH_ROWS = 1` bounds the FX work per invocation, and `COMPRESSION = NONE` keeps the proxy body directly readable.


In [ ]:
USE ROLE ADV_LAB;
USE SECONDARY ROLES NONE;
USE WAREHOUSE ADV_WH;
USE DATABASE ADV_ORDERS;
USE SCHEMA OPS;


In [ ]:
fx_function_ddl = f"""
CREATE OR REPLACE EXTERNAL FUNCTION ADV_ORDERS.OPS.GET_FX_RATE(BASE_CURRENCY VARCHAR, QUOTE_CURRENCY VARCHAR)
RETURNS VARIANT
RETURNS NULL ON NULL INPUT
VOLATILE
API_INTEGRATION = {integration_name}
MAX_BATCH_ROWS = 1
COMPRESSION = NONE
AS {sql_literal(fx_url)}
"""
print(fx_function_ddl)
session.sql(fx_function_ddl).collect()
print("FX external function created.")


**NULL checkpoint:** Run the next cell. It should return TRUE without calling the remote service.


In [ ]:
SELECT ADV_ORDERS.OPS.GET_FX_RATE(NULL, 'EUR') IS NULL AS NULL_INPUT_PASSED;


**Fetch once, then inspect:** Run the next cell once in this notebook session. The table stores the complete response, then derives the `RATE` and `RATE_DATE` columns locally from that stored object. This avoids an intentional FX call per order; Snowflake or the platform can retry remote requests. Do not rerun this cell without dropping the table first. The dedicated temporary name must not already be used by a permanent course object; temporary tables can shadow permanent names. This table is session-scoped, not a pipeline object.


In [ ]:
%%sql
CREATE TEMPORARY TABLE ADV_ORDERS.OPS.LAB21_FX_USD_EUR AS
SELECT ADV_ORDERS.OPS.GET_FX_RATE('USD', 'EUR') AS FX_RESPONSE;
ALTER TABLE ADV_ORDERS.OPS.LAB21_FX_USD_EUR ADD COLUMN RATE NUMBER(38, 12), RATE_DATE DATE;
UPDATE ADV_ORDERS.OPS.LAB21_FX_USD_EUR
SET RATE = FX_RESPONSE:rate::NUMBER(38, 12), RATE_DATE = FX_RESPONSE:date::DATE;
SELECT FX_RESPONSE, RATE, RATE_DATE FROM ADV_ORDERS.OPS.LAB21_FX_USD_EUR;


**CHECKPOINT:** exactly one row with a USD/EUR response, positive rate and rate date. If the call fails, fix the error before continuing; do not insert an invented rate. The response date and rate vary.

Run the separate read-only query below. It cross joins that single stored rate to the existing orders. `DEMO_ORDER_RISK` still uses the original threshold (`> 100` is REVIEW, `100` is OK, NULL is UNKNOWN). FX is not called per order; the risk external function is still invoked for the order rows.


In [ ]:
SELECT O.ORDER_ID,
       O.AMOUNT AS ORIGINAL_USD_AMOUNT,
       ADV_ORDERS.OPS.DEMO_ORDER_RISK(O.AMOUNT::FLOAT) AS RISK_LABEL,
       FX.RATE AS USD_TO_EUR_RATE,
       FX.RATE_DATE,
       ROUND(O.AMOUNT * FX.RATE, 2) AS EUR_AMOUNT
FROM ADV_ORDERS.SILVER.ORDERS_CURRENT AS O
CROSS JOIN ADV_ORDERS.OPS.LAB21_FX_USD_EUR AS FX
ORDER BY O.ORDER_ID;


**Optional FX table cleanup:** After inspecting the conversion results, run the next cell only if you created the FX temporary table in this same session. This drops the temporary table before the common cleanup; it does not drop the external function. If you skip the extension, skip this cell. A temporary table also disappears automatically when its session ends. Never run this cell against an unrelated permanent table of the same name.


In [ ]:
DROP TABLE IF EXISTS ADV_ORDERS.OPS.LAB21_FX_USD_EUR;


### STEP 15 | Clean up Snowflake objects in the notebook
Run cleanup **after** verifying all chosen demo checkpoints. The following SQL cell removes the two lab external functions and API integration. Run it when finished; it does not remove course tables, roles, or the warehouse.

Dropping Snowflake objects does **not** delete AWS resources. Complete the AWS console cleanup below as well.


In [ ]:
%%sql
-- Execute only after completing all checks.
USE ROLE ACCOUNTADMIN;
DROP FUNCTION IF EXISTS ADV_ORDERS.OPS.GET_FX_RATE(VARCHAR, VARCHAR);
DROP FUNCTION IF EXISTS ADV_ORDERS.OPS.DEMO_ORDER_RISK(FLOAT);
DROP INTEGRATION IF EXISTS ORDER_RISK_API_INTEGRATION;

In [ ]:
print("AWS cleanup targets:")
print("Region:", aws_region)
print("API Gateway REST API:", api_name, "| API ID:", api_id)
print("Lambda:", lambda_name)
print("Invocation IAM role:", aws_role_name)
print("Inline policy:", inline_policy_name)
print("CloudWatch log group:", "/aws/lambda/" + lambda_name)


### STEP 16A | Inspect AWS logs and clean up in one visit
Before deletion, open the configured Lambda → **Monitor → View CloudWatch logs** to inspect invocation entries. Then complete all AWS cleanup in this visit.

1. **API Gateway:** select the REST API by the printed name **and API ID**, then delete that API using its actions menu and confirm. This removes its deployment stages and the optional `/fx` resource as well; `/fx` has no separate AWS resource to delete.
2. **Lambda:** before deleting, record the auto-created execution role under **Configuration → Permissions**. Delete the configured Lambda through **Actions → Delete**.
3. **IAM:** delete the dedicated invocation role and its inline policy. Delete the auto-created Lambda execution role only after confirming it belongs solely to this demonstration. Do not delete AWS-managed policies or a shared execution role. Remove a dedicated customer-managed logging policy only if it is unused and was created for this Lambda.
4. **CloudWatch → Logs → Log groups:** delete `/aws/lambda/<configured Lambda name>` if you do not need its logs. Lambda deletion does not automatically remove retained logs.
5. Confirm the API, Lambda and dedicated IAM roles are gone. If no other course work is running, use the course shutdown script to suspend course compute and background workloads; close the notebook session separately.

**To rehearse from scratch:** complete both Snowflake and AWS cleanup, restart the notebook kernel/session, start at Step 1. The new API gets a new ID and URL: update Step 8. Recreate the integration and regenerate the trust policy; do not reuse the old external ID. Execute each cell in order, pausing for the AWS console steps. Keep cleanup until the end.

### References
- [Snowflake: create an external function using the AWS console](https://docs.snowflake.com/en/sql-reference/external-functions-creating-aws-ui)
- [Snowflake: Lambda setup and batch data format](https://docs.snowflake.com/en/sql-reference/external-functions-creating-aws-ui-remote-service)
- [Snowflake: API integration](https://docs.snowflake.com/en/sql-reference/external-functions-creating-aws-common-api-integration)
- [Frankfurter: v2 rate API](https://frankfurter.dev/)
- [Snowflake: external function data format](https://docs.snowflake.com/en/sql-reference/external-functions-data-format)
- [AWS: Lambda proxy integration](https://docs.aws.amazon.com/apigateway/latest/developerguide/set-up-lambda-proxy-integrations.html)
- [AWS: API Gateway security policies](https://docs.aws.amazon.com/apigateway/latest/developerguide/apigateway-security-policies-list.html)


In [ ]:
print(f"""# AWS external-function lab cleanup
AWS_ACCOUNT_ID={aws_account_id}
AWS_DEFAULT_REGION={aws_region}
AWS_API_ID={api_id}
AWS_API_NAME={api_name}
AWS_LAMBDA_FUNCTION={lambda_name}
AWS_INVOKE_ROLE={aws_role_name}
AWS_INVOKE_POLICY={inline_policy_name}
AWS_LAMBDA_EXECUTION_ROLE={lambda_execution_role}
""")

### ### STEP 16B | Clean up AWS resources using local Python

Run these steps on your workstation, after executing the Snowflake cleanup SQL above.

#### 1. Prepare the local `.env` file

Copy the settings printed by the preceding notebook cell into the `.env` file at the **root of your local course repository**.

Add your AWS credentials to that same file:

```dotenv
AWS_ACCESS_KEY_ID=YOUR_ACCESS_KEY_ID
AWS_SECRET_ACCESS_KEY=YOUR_SECRET_ACCESS_KEY
```

The combined AWS section should contain:

```dotenv
AWS_ACCESS_KEY_ID=YOUR_ACCESS_KEY_ID
AWS_SECRET_ACCESS_KEY=YOUR_SECRET_ACCESS_KEY

AWS_ACCOUNT_ID=YOUR_12_DIGIT_ACCOUNT_ID
AWS_DEFAULT_REGION=YOUR_API_REGION
AWS_API_ID=YOUR_API_ID
AWS_API_NAME=snowflake-order-risk-api
AWS_LAMBDA_FUNCTION=snowflake-order-risk-demo
AWS_INVOKE_ROLE=SnowflakeOrderRiskInvokeRole
AWS_INVOKE_POLICY=InvokeOrderRiskDemo
AWS_LAMBDA_EXECUTION_ROLE=YOUR_LAMBDA_EXECUTION_ROLE_NAME
```

Replace the placeholders with your values. Use the region in your API Gateway Invoke URL. For the Lambda execution role, use the **role name**, not its ARN.

Keep existing Snowflake settings in `.env`. Replace existing AWS entries rather than adding duplicates. Keep credentials local; do not paste them into the notebook or commit them to Git.

`AWS_SESSION_TOKEN` is needed only when using temporary AWS credentials.

#### 2. Open a local terminal

Open PowerShell or your Python environment's terminal. Change to the local course repository root—the folder containing `.env`, `Day1`, `Day2`, and `Day3`.

If you use a virtual environment, activate it first.

Install the cleanup dependencies:

```powershell
python -m pip install -r Day3/21-requirements-cleanup.txt
```

#### 3. Preview the cleanup

```powershell
python Day3/21-cleanup_aws.py
```

Without `--execute`, the script only lists planned deletions.

Check the account, region, API ID, function, roles, and log group. The script checks for mismatches before deleting anything.

If `AWS_LAMBDA_EXECUTION_ROLE` is blank, the Lambda execution role is retained.

#### 4. Execute cleanup

After reviewing the preview, run:

```powershell
python Day3/21-cleanup_aws.py --execute
```

The script removes the specified API, Lambda function, log group, and dedicated IAM roles. It removes inline policies and detaches managed policies from deleted roles; managed policy objects themselves are retained.

The credentials must have permission to inspect and delete these resources. Checking whether a role is shared also requires listing enabled regions and Lambda functions in those regions.

#### 5. Verify completion

Run the preview command again. For a complete cleanup of the selected resources, it should report:

```text
No matching resources remain.
```

If execution stops partway through, resolve the reported problem and rerun. Resources already deleted are skipped.

**Region troubleshooting:** existing environment variables take precedence over `.env`. If PowerShell contains an outdated region value, clear it and rerun:

```powershell
Remove-Item Env:AWS_DEFAULT_REGION -ErrorAction SilentlyContinue
```